In [16]:
library(dplyr)
library(tidyr)
library(data.table)


Attaching package: ‘data.table’


The following object is masked from ‘package:GenomicRanges’:

    shift


The following object is masked from ‘package:IRanges’:

    shift


The following objects are masked from ‘package:S4Vectors’:

    first, second


The following objects are masked from ‘package:dplyr’:

    between, first, last




In [18]:
# map gene id from one type to another
mapGeneIDs <- function(query, query_type,
                        output_types) {
  library(org.Hs.eg.db)
  mapped = AnnotationDbi::select(org.Hs.eg.db,
                  keys = query,
                  columns = output_types,
                  keytype = query_type)
  return(mapped)
}

In [2]:
setwd("~/projects/lung_scMultiomics/analysis/")

In [7]:
library(openxlsx)
summary_tbl <- readxl::read_xlsx("../output/E2G_analysis/asthma_candidate_risk_genes_43CREs.xlsx")
pops_p2g <- readRDS("../data/plotting_inputs/E2G_PoPs_by_genes.RDS")
eqtl_list <- readRDS("../output/E2G_analysis/natri2023_eQTLs.RDS")
genes <- unique(unlist(lapply(summary_tbl$LinkedGene, function(i) {strsplit(i, ",")[[1]]})))
gene_tbl <- 
  summary_tbl %>% 
    separate_rows(LinkedGene, sep = ",") %>%
    dplyr::select(-c(Peak, GeneScore, NewCREs, CandidateGenes, susie_pips)) %>%
    distinct()
prior_tbl <- 
    summary_tbl %>%
    separate_rows(CandidateGenes,sep = ",") %>%
    filter(!is.na(CandidateGenes)) %>%
    dplyr::select(CandidateGenes, LinkedGene, snps) %>%
    distinct()

In [19]:
eqtl_list <-
  lapply(c("Bcells", "CD4", "CD8NKT"), function(i) {
    eqtl <- 
      fread(sprintf("~/cluster/projects/u19_multiomics/other_data/Natri2023/immune_%s.tsv", i))
    # LFSR <= 0.05 in at least one cell type
    eqtl <- eqtl[!is.na(eqtl$snp_rsid) & eqtl$lfsr <= 0.05, ]
    eqtlDF <- do.call(
      rbind,
      lapply(eqtl$snp_id, function(i) {
        words = strsplit(i, "_")[[1]]
        return(words)
      }))
    eqtl_gr <- GRanges(seqnames = paste0("chr", eqtlDF[, 1]),
                       IRanges(start = eqtlDF[, 2]),
                       gene = eqtl$feature_id,
                       rsid = eqtl$snp_rsid,
                       lfsr = eqtl$lfsr)
    
    mapped_genes <- mapGeneIDs(query = eqtl_gr$gene, 
                               query_type = "ENSEMBL", 
                               output_types = "SYMBOL")
    eqtl_gr$targetGene <- mapped_genes$SYMBOL[match(eqtl_gr$gene, mapped_genes$ENSEMBL)]
    eqtl_gr <- eqtl_gr[!is.na(eqtl_gr$targetGene)]
    return(eqtl_gr)
  })
names(eqtl_list) <- c("B", "CD4_T", "CD8_T")

Loading required package: AnnotationDbi

Loading required package: Biobase

Welcome to Bioconductor

    Vignettes contain introductory material; view with
    'browseVignettes()'. To cite Bioconductor, see
    'citation("Biobase")', and for packages 'citation("pkgname")'.



Attaching package: ‘AnnotationDbi’


The following object is masked from ‘package:dplyr’:

    select




'select()' returned many:many mapping between keys and columns

'select()' returned many:many mapping between keys and columns

'select()' returned many:many mapping between keys and columns



In [66]:
prior_tbl$PoPS <- pops_p2g$pops[match(prior_tbl$CandidateGenes, pops_p2g$SYMBOL)]

In [67]:
prior_tbl

CandidateGenes,LinkedGene,snps,Lung_eGene,Asthma_eGene,risk_genes,PoPS
<chr>,<chr>,<chr>,<lgl>,<lgl>,<lgl>,<dbl>
IL2RA,"IL15RA,IL2RA",rs12722502,FALSE,FALSE,TRUE,2.73284479
PRKCQ,"IL15RA,IL2RA",rs12722502,FALSE,TRUE,TRUE,1.72350993
PRKCQ,PRKCQ,rs2255088,FALSE,TRUE,TRUE,1.72350993
IL2RA,PRKCQ,rs2255088,FALSE,FALSE,TRUE,2.73284479
CXCR5,"CXCR5,DDX6","rs12365699,rs4936444",FALSE,FALSE,FALSE,0.54263983
OVOL1,OVOL1,rs10791824,FALSE,FALSE,FALSE,0.46379917
LRRC32,LRRC32,"rs55646091,rs11236797",FALSE,FALSE,FALSE,1.15860517
RPS26,"ERBB3,IKZF4,RPS26","rs705704,rs705705",FALSE,TRUE,FALSE,1.23469253
STAT6,LRP1,rs12368672,TRUE,FALSE,FALSE,NA


In [21]:
eqtl_pairs <-
  lapply(prior_tbl$CandidateGenes, function(i) {
    pairs <-c()
    for(ct in names(eqtl_list)) {
      if(i %in% eqtl_list[[ct]]$targetGene) {
        rows = 
          data.frame(mcols(eqtl_list[[ct]])) %>% 
            filter(targetGene == i) %>%
              mutate(CandidateGenes = targetGene,
                     snps = rsid) %>%
              dplyr::select(rsid, targetGene)
        pairs <- rbind(pairs,
          data.frame(
            "CellType" = ct,
            rows))
      }
    }
    return(pairs)
  })
eqtl_pairs <- do.call(rbind, eqtl_pairs[!sapply(eqtl_pairs, is.null)])

In [35]:
gene_tbl$Lung_eGene <- gene_tbl$LinkedGene %in% eqtl_pairs$targetGene
prior_tbl$Lung_eGene <- prior_tbl$CandidateGenes %in% eqtl_pairs$targetGene

In [48]:
riskGenes_set1 <- fread("~/cluster/projects/u19_multiomics/other_data/weilong/TableS5_1_asthma-risk-genes_ALOFT__TableS12.txt.gz")
riskGenes_set2 <- fread("~/cluster/projects/u19_multiomics/other_data/weilong/TableS5_2_asthma-risk-genes_GTEx__TableS13.txt.gz")

In [49]:
# The PCGs are defined as those with an FDR < 10% from INTACT - joint usage of TWAS and colocalization
quantile(riskGenes_set2$FDR_intact)

0%       25%       50%       75%      100% 
0.0000000 0.8729932 0.9340100 0.9554698 0.9753162

In [53]:
head(riskGenes_set1 %>% filter(FDR_intact < 0.1))

Gene,symbol,genetic_variant,chromosome,chr_pos_grch38,chr_pos_grch37,PIP,pval_eqtl,peaking_d,pval_twas,zscore_twas,FDR_twas,GLCP,PCG,FDR_intact
<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
ENSG00000073605,GSDMB,17:38051348:A:G;rs8067378,17,17_39895095,17_38051348,0.9993510,4.42171e-34,0,1.032e-98,-21.08763,9.102756e-95,0.88220,1,0
ENSG00000112182,BACH2,6:91006227:G:A;rs1504215,6,6_90296508,6_91006227,0.0662612,1.32959e-01,3,1.922e-34,-12.23907,2.119125e-31,0.11130,1,0
ENSG00000172057,ORMDL3,17:38089317:A:AAAGG;rs140683487,17,17_39933064,17_38089317,0.9501850,9.71619e-24,0,7.714e-31,11.54626,7.560149e-28,0.80950,1,0
ENSG00000131507,NDFIP1,5:141489921:C:T;rs12653866,5,5_142110356,5_141489921,0.1532790,5.76374e-06,3,1.656e-25,10.43836,1.270152e-22,0.97370,1,0
ENSG00000125245,GPR18,13:99910263:C:T;rs3759444,13,13_99258009,13_99910263,0.0349365,1.49488e-02,1,2.069e-25,-10.41714,1.520801e-22,0.07594,1,0
ENSG00000134882,UBAC2,13:99852753:A:G;rs2296860,13,13_99200499,13_99852753,0.1820240,1.14869e-02,2,8.176e-25,10.28567,5.769313e-22,0.12100,1,0


In [54]:
# asthma eGene datasets
eGenes_set1 <- fread("~/cluster/projects/u19_multiomics/other_data/weilong/TableS4_3_eGene_ALOFT_dap__TableS10.txt.gz")
eGenes_set2 <- fread("~/cluster/projects/u19_multiomics/other_data/weilong/TableS4_4_eGene_GTEx_dap__TableS11.txt.gz")
riskGene_set1 <- fread("~/cluster/projects/u19_multiomics/other_data/weilong/TableS5_1_asthma-risk-genes_ALOFT__TableS12.txt.gz")
riskGenes_set2 <- fread("~/cluster/projects/u19_multiomics/other_data/weilong/TableS5_2_asthma-risk-genes_GTEx__TableS13.txt.gz")

eGenes_combined <- rbind(
  data.frame(study = "ALOFT", eGenes_set1),
  data.frame(study = "GTEx", eGenes_set2)
)
riskGenes_combined <- unique(
    c(riskGenes_set1$symbol[riskGenes_set1$FDR_intact < 0.1],
      riskGenes_set2$symbol[riskGenes_set1$FDR_intact < 0.1])
    )
mapping <- 
  mapGeneIDs(eGenes_combined$gene, 
             query_type  = "ENSEMBL", 
             output_types = "SYMBOL")
eGenes_combined$eGenes <- 
  mapping$SYMBOL[match(eGenes_combined$gene, mapping$ENSEMBL)]

eGenes_combined %>% filter(eGenes %in% genes)

'select()' returned 1:many mapping between keys and columns



study,gene,lfdr,FDR,eGenes
<chr>,<chr>,<dbl>,<dbl>,<chr>
ALOFT,ENSG00000073584,7.3307e-29,2.811162e-30,SMARCE1
ALOFT,ENSG00000197728,1.4197e-27,5.093644e-29,RPS26
ALOFT,ENSG00000065361,3.9589e-09,1.721136e-10,ERBB3
ALOFT,ENSG00000148399,4.3954e-07,2.053128e-08,DPH7
ALOFT,ENSG00000134470,2.1376e-05,1.297504e-06,IL15RA
ALOFT,ENSG00000134987,1.4580e-04,8.133146e-06,WDR36
ALOFT,ENSG00000180902,5.4611e-04,3.337787e-05,D2HGDH
ALOFT,ENSG00000197070,5.4649e-02,4.706086e-03,ARRDC1
ALOFT,ENSG00000141655,1.1080e-01,1.054306e-02,TNFRSF11A


In [64]:
saveRDS(prior_tbl, "../output/E2G_analysis/prior_study_genes_evidence.RDS")

In [56]:
prior_tbl$Asthma_eGene <- 
  prior_tbl$CandidateGenes %in% eGenes_combined$eGenes[eGenes_combined$study == "ALOFT"]
gene_tbl$Asthma_eGene <- 
  gene_tbl$LinkedGene %in% eGenes_combined$eGenes[eGenes_combined$study == "ALOFT"]

In [59]:
prior_tbl$CandidateGenes %in% riskGenes_combined

[1]  TRUE  TRUE  TRUE  TRUE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
[13] FALSE FALSE FALSE FALSE FALSE FALSE FALSE  TRUE FALSE FALSE FALSE FALSE
[25] FALSE FALSE  TRUE  TRUE

In [60]:
prior_tbl$risk_genes <- 
  prior_tbl$CandidateGenes %in% riskGenes_combined

In [61]:
gene_tbl$risk_genes <- 
  gene_tbl$LinkedGene %in% riskGenes_combined

In [62]:
prior_tbl

CandidateGenes,LinkedGene,snps,Lung_eGene,Asthma_eGene,risk_genes
<chr>,<chr>,<chr>,<lgl>,<lgl>,<lgl>
IL2RA,"IL15RA,IL2RA",rs12722502,FALSE,FALSE,TRUE
PRKCQ,"IL15RA,IL2RA",rs12722502,FALSE,TRUE,TRUE
PRKCQ,PRKCQ,rs2255088,FALSE,TRUE,TRUE
IL2RA,PRKCQ,rs2255088,FALSE,FALSE,TRUE
CXCR5,"CXCR5,DDX6","rs12365699,rs4936444",FALSE,FALSE,FALSE
OVOL1,OVOL1,rs10791824,FALSE,FALSE,FALSE
LRRC32,LRRC32,"rs55646091,rs11236797",FALSE,FALSE,FALSE
RPS26,"ERBB3,IKZF4,RPS26","rs705704,rs705705",FALSE,TRUE,FALSE
STAT6,LRP1,rs12368672,TRUE,FALSE,FALSE


In [63]:
gene_tbl

LinkedGene,trait,snps,Lung_eGene,Asthma_eGene,risk_genes
<chr>,<chr>,<chr>,<lgl>,<lgl>,<lgl>
IL15RA,coa,rs12722502,FALSE,TRUE,FALSE
IL2RA,coa,rs12722502,FALSE,FALSE,TRUE
PRKCQ,coa,rs2255088,FALSE,TRUE,TRUE
CXCR5,coa,"rs12365699,rs4936444",FALSE,FALSE,FALSE
DDX6,coa,"rs12365699,rs4936444",FALSE,FALSE,TRUE
OVOL1,coa,rs10791824,FALSE,FALSE,FALSE
LRRC32,coa,"rs55646091,rs11236797",FALSE,FALSE,FALSE
ERBB3,coa,"rs705704,rs705705",FALSE,TRUE,FALSE
IKZF4,coa,"rs705704,rs705705",FALSE,FALSE,FALSE
